# German Credit: Calibrating Scorecard Points to Probabilities

This notebook shows how to calibrate scorecard points (LR or SHAP-based) to well-calibrated probabilities using `scorecardpl.calibration`.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import polars as pl
from sklearn.ensemble import RandomForestClassifier

from scorecardpl import (
    split_df, var_filter, woebin, woebin_ply,
    scorecard, scorecard_ply, perf_eva,
    scorecard_shap,
)
from scorecardpl.calibration import (
    calibrate_scorecard_from_data, fit_points_calibrator,
)


## Load and prepare German Credit

In [ ]:
uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
cols = [
    'Status', 'Duration', 'CreditHistory', 'Purpose', 'CreditAmount',
    'Savings', 'Employment', 'InstallmentRate', 'PersonalStatusSex', 'OtherDebtors',
    'ResidenceSince', 'Property', 'Age', 'OtherInstallmentPlans', 'Housing',
    'ExistingCredits', 'Job', 'Liables', 'Telephone', 'ForeignWorker', 'class'
]
df_pd = pd.read_csv(uci_url, sep=' ', header=None, names=cols)
df_pd['y'] = (df_pd['class'] == 2).astype(int)
df_pd = df_pd.drop(columns=['class'])
df_pl = pl.from_pandas(df_pd)
y = 'y'
train, valid = split_df(df_pl, y=y, test_size=0.3, random_state=42)
train = var_filter(train, y=y)
bins = woebin(train, y=y, x=[c for c in train.columns if c != y],
                bins=6, method='chi2', chi2_params={'init_bins': 60}, monotonic='auto', cat_max_bins=5)


## 1) Logistic scorecard + calibration

In [ ]:
train_w = woebin_ply(train, bins)
valid_w = woebin_ply(valid, bins)
sc_lr = scorecard(bins, y=y, data=train_w)
scores_valid_lr = scorecard_ply(valid_w, sc_lr.points_map).to_numpy()
# Raw LR probabilities (from WOE features)
X_valid = valid_w.select([c for c in valid_w.columns if c.endswith('_woe')]).to_numpy()
proba_raw = sc_lr.model.predict_proba(X_valid)[:, 1]
# Calibrate points -> probability (logistic)
cal_lr = calibrate_scorecard_from_data(bins, sc_lr.points_map, y=y, data=train)
proba_cal_lr = cal_lr.points_to_proba(scores_valid_lr)
perf_raw = perf_eva(valid[y], proba_raw)
perf_cal = perf_eva(valid[y], proba_cal_lr)
{'raw': perf_raw, 'calibrated': perf_cal}


## 2) SHAP-based scorecard + calibration

In [ ]:
est = RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42)
sc_shap = scorecard_shap(bins=bins, y=y, data=train, estimator=est, shap_sample_n=2000)
# Uncalibrated estimator probabilities (from RF on WOE features)
proba_uncal = sc_shap.predict_proba(valid, bins)
# Calibrate points -> probability (logistic)
cal_shap = calibrate_scorecard_from_data(bins, sc_shap.points_map, y=y, data=train)
proba_cal_shap = sc_shap.predict_proba(valid, bins, calibrator=cal_shap)
perf_uncal = perf_eva(valid[y], proba_uncal)
perf_cal = perf_eva(valid[y], proba_cal_shap)
{'uncalibrated': perf_uncal, 'calibrated': perf_cal}
